# Spindle — Composite Domain

**v1.3.0** introduces `CompositeDomain` for generating data that spans multiple verticals in a single run, with cross-domain foreign keys automatically enforced.

Use this when your Fabric solution has tables from more than one business domain — for example:
- **HR + Retail**: employees who are also loyalty customers
- **Healthcare + Insurance**: patients cross-referenced to policyholders
- **Financial + Retail + HR**: full enterprise data platform

The `SharedEntityRegistry` maps real-world concepts (`PERSON`, `LOCATION`, `ORGANIZATION`, `CALENDAR`) to their domain-specific tables, so Spindle can wire FKs automatically.

In [ ]:
%pip install sqllocks-spindle --quiet

In [ ]:
from sqllocks_spindle.domains.retail.retail import RetailDomain
from sqllocks_spindle.domains.hr.hr import HrDomain
from sqllocks_spindle.domains.financial.financial import FinancialDomain
from sqllocks_spindle.domains.healthcare.healthcare import HealthcareDomain
from sqllocks_spindle.domains.composite import CompositeDomain
from sqllocks_spindle.domains.shared_registry import SharedEntityRegistry, SharedConcept
from sqllocks_spindle.engine.generator import Spindle

print('Spindle loaded. Available SharedConcept values:')
for c in SharedConcept:
    print(f'  {c.value}')

## 1. Retail + HR — Shared PERSON

Generate retail customers and HR employees where the HR employee table is the canonical source of truth for person data. Retail customer records carry a foreign key back to `hr.employee`.

In [ ]:
composite_rh = CompositeDomain(
    domains=[RetailDomain(), HrDomain()],
    shared_entities={
        'person': {
            'primary': 'hr.employee',
            'links': {
                'retail': 'customer.employee_id',
            },
        },
    },
)

result_rh = Spindle().generate(domain=composite_rh, scale='fabric_demo', seed=42)
print(result_rh)

In [ ]:
print('Tables generated:')
for name, df in result_rh.tables.items():
    print(f'  {name:<30} {len(df):>6,} rows')

In [ ]:
# Verify cross-domain FK: retail customer.employee_id -> hr employee.employee_id
tables = result_rh.tables

if 'employee' in tables and 'customer' in tables:
    emp_ids = set(tables['employee']['employee_id'].dropna().unique())
    cust_fk = tables['customer'].get('employee_id')
    if cust_fk is not None:
        linked  = cust_fk.dropna()
        orphans = linked[~linked.isin(emp_ids)]
        print(f'HR employees:             {len(emp_ids):,}')
        print(f'Retail->HR FK references: {len(linked):,}')
        print(f'Orphan references:        {len(orphans):,}  (must be 0)')
    else:
        print('employee_id FK not present in customer table (depends on domain config)')
else:
    print(f'Available tables: {sorted(tables.keys())}')

## 2. Retail + HR + Financial — Three-domain composite

Add a third domain. Shared PERSON (HR → Retail + Financial) and shared LOCATION (Retail store → Financial branch).

In [ ]:
composite_3 = CompositeDomain(
    domains=[RetailDomain(), HrDomain(), FinancialDomain()],
    shared_entities={
        'person': {
            'primary': 'hr.employee',
            'links': {
                'retail':    'customer.employee_id',
                'financial': 'customer.employee_id',
            },
        },
        'location': {
            'primary': 'retail.store',
            'links': {
                'financial': 'branch.store_id',
            },
        },
    },
)

result_3 = Spindle().generate(domain=composite_3, scale='fabric_demo', seed=42)
print(f'Tables: {len(result_3.tables)}')
print(f'Total rows: {sum(len(df) for df in result_3.tables.values()):,}')
print(f'\nChild domains: {[d.__class__.__name__ for d in composite_3.child_domains]}')

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'table': k, 'rows': len(v), 'columns': len(v.columns)}
    for k, v in result_3.tables.items()
]).sort_values('rows', ascending=False)
summary

## 3. Auto-registry

When you don't provide `shared_entities`, `SharedEntityRegistry` applies built-in default concept mappings automatically. Great for quick multi-domain exploration.

In [ ]:
registry = SharedEntityRegistry()

composite_auto = CompositeDomain(
    domains=[RetailDomain(), HealthcareDomain()],
    registry=registry,
)

result_auto = Spindle().generate(domain=composite_auto, scale='fabric_demo', seed=42)
print(result_auto)
print('\nTables:', sorted(result_auto.tables.keys()))

## 4. Write composite data to Fabric Delta Tables

All tables from a composite domain write to Lakehouse exactly like single-domain tables — use `spark.createDataFrame()` as normal.

In [ ]:
# This cell runs in a Fabric Spark notebook
# spark is pre-wired by the Fabric runtime

for table_name, df in result_3.tables.items():
    spark_df = spark.createDataFrame(df)
    (
        spark_df.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .saveAsTable(f'composite_{table_name}')
    )
    print(f'  Wrote composite_{table_name} ({len(df):,} rows)')

print('\nAll composite domain tables written to Lakehouse Delta.')

In [ ]:
# Cross-domain Spark SQL query: employees who are also retail customers
spark.sql("""
    SELECT
        e.employee_id,
        e.first_name,
        e.last_name,
        e.department,
        COUNT(o.order_id) AS order_count,
        SUM(o.total_amount) AS lifetime_value
    FROM composite_employee e
    JOIN composite_customer c ON c.employee_id = e.employee_id
    JOIN composite_order o    ON o.customer_id = c.customer_id
    GROUP BY e.employee_id, e.first_name, e.last_name, e.department
    ORDER BY lifetime_value DESC
    LIMIT 10
""").show()